# Pre-Morph Before Symbol and Voronoi Layouts

The symbol and Voronoi layouts start from the positions of the input geometries. When the areas of those geometries have nothing to do with the data (the states of the US, sized by land area), a region with a large value gets a symbol or a cell that does not fit where its geometry is, and its neighbors have to give way. A flow cartogram of the same data moves the regions to where they would be if their areas were proportional to the data, so a layout that starts from it has less to correct.

This guide pre-morphs the input with a flow cartogram and compares the layouts on the raw and on the pre-morphed geometries: how long they take to converge and what they look like. The Voronoi cartogram has the option `premorph=True` that does this for you; the symbol layouts take the morphed geometries as their input. The effect is not the same for every layout, and the measurements below say where it helps.

## Import Libraries

In [ ]:
import matplotlib.pyplot as plt

import carto_flow.data as examples
import carto_flow.flow_cartogram as flow
import carto_flow.symbol_cartogram as smb
import carto_flow.voronoi_cartogram as vor

## Load Geographic Data

The contiguous US states with their population.

In [ ]:
gdf = examples.load_us_census(population=True)

## Pre-Morph with a Flow Cartogram

The flow cartogram sizes the regions by population. `to_geodataframe()` returns a GeoDataFrame with the morphed geometries and the columns of the input, which can be used wherever the input is used.

In [ ]:
cartogram = flow.create_flow_cartogram(
    gdf, "Population", options=flow.MorphOptions(area_scale=1e-6, show_progress=False)
)
print(cartogram.status.name)

premorphed = cartogram.to_geodataframe()

## Symbol Layouts

The same three layouts are run on the raw and on the pre-morphed geometries. The `centroid` layout places a symbol at each centroid, and `packing` and `flow_density` move the symbols until they no longer overlap.

In [ ]:
layouts = ["centroid", "packing", "flow_density"]
sources = {"raw geometry": gdf, "flow pre-morph": premorphed}

results = {
    (source, layout): smb.create_symbol_cartogram(data, "Population", layout=layout, show_progress=False)
    for source, data in sources.items()
    for layout in layouts
}

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6.5))
for row, (source, data) in enumerate(sources.items()):
    data.plot(column="Population", ax=axes[row, 0], linewidth=0.3)
    axes[row, 0].set_ylabel(source)
    for column, layout in enumerate(layouts, start=1):
        results[source, layout].plot(facecolor="Population", legend=False, ax=axes[row, column])
for ax, title in zip(axes[0], ["Input", *layouts], strict=True):
    ax.set_title(title)
for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
    for side in ax.spines.values():
        side.set_visible(False)

How long each layout took to converge, on the raw and on the pre-morphed geometries:

In [ ]:
print(f"{'layout':<14}{'input':<16}{'iterations':>10}  converged")
for (source, layout), result in results.items():
    layout_result = result.layout_result
    print(f"{layout:<14}{source:<16}{layout_result.iterations:>10}  {layout_result.converged}")

The `centroid` layout is not changed by the pre-morph in any way that matters: it puts a symbol at each centroid. The two layouts that resolve overlaps need fewer iterations when they start from the pre-morphed geometries, and the symbols stay closer to where the flow cartogram has put their regions. Treat the saving as modest (about a fifth to a quarter of the iterations here). It is paid for by one flow morph, which takes less than a second here.

## Tile Maps

The mosaic layout, which gives each region a number of tiles, does the pre-morph itself: `MosaicLayout(morph=True)` is the default, so it should be given the raw geometries. Pre-morphing its input as well would morph the regions twice. The grid layout (one tile per region) starts from the positions of the geometries like the symbol layouts above, so it can be given the pre-morphed geometries in the same way; the effect on the assignment of the tiles was not measured here.

## Voronoi Cartogram

`premorph=True` runs the flow cartogram inside `create_voronoi_cartogram`, starts the cells from the morphed regions, and keeps the morphed cartogram in the `premorph` attribute of the result.

In [ ]:
kwargs = {
    "weights": "Population",
    "backend": vor.RasterBackend(resolution=256),
    "options": vor.VoronoiOptions(n_iter=300, area_cv_tol=0.1, show_progress=False),
}
raw_cells = vor.create_voronoi_cartogram(gdf, **kwargs)
premorphed_cells = vor.create_voronoi_cartogram(gdf, premorph=True, **kwargs)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (title, result) in zip(axes, [("raw geometry", raw_cells), ("premorph=True", premorphed_cells)], strict=True):
    result.plot(facecolor="Population", legend=False, ax=ax)
    ax.set_title(f"{title}: {result.metrics['n_iterations']} iterations")

The Voronoi relaxation needs far fewer iterations from the pre-morphed regions, and the cells keep the outline of the flow cartogram instead of the outline of the input.

## When to Use It

- **Voronoi cartogram.** Use `premorph=True` when the cells should follow a flow cartogram, or to save iterations.
- **`packing` and `flow_density`.** Pre-morph when the symbols should stay close to the flow cartogram. The saving in iterations is modest.
- **`centroid` and the mosaic layout.** No pre-morph: the first does not need it and the second morphs by itself (`morph=True`).
- **Cost.** One flow morph; see [Diagnose and Improve Convergence](diagnose-convergence.ipynb) when it does not converge.